Reference: https://j2rooong.tistory.com/entry/Pytorch-Transformer-Architecture-구현하기

In [ ]:
import torch
import torch.nn as nn
import math

# ----------------------------------------------------------------
# 1. Input Embedding
# ----------------------------------------------------------------
class InputEmbeddings(nn.Module):
    """
    create an input embedding
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        # table that should be trained, should be updated 
        self.embedding = nn.Embedding(vocab_size, d_model)  

    def forward(self, x):
        # matrix * constant
        # To prevent a positional embedding from shadowing the input embedding
        return self.embedding(x) * math.sqrt(self.d_model)
    
# ----------------------------------------------------------------
# 2. Positional Encoding
# ----------------------------------------------------------------
class PositionalEncoding(nn.Module):
    # This method doesnt return anything (None)
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None: 
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        
        # 2D tensor with 0s
        pe = torch.zero(seq_len, d_model)   
        # 2D tensor (seq_len, 1)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(dim=1)

        _2i = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float))

        # Giving each word an unique barcode
        # A clock with 512 needles
        # 0::2 -> from 0 to the end, with a step of 2
        pe[:, 0::2] = torch.sin(position/10000**(_2i/d_model))
        pe[:, 1::2] = torch.cos(position/10000**(_2i/d_model))

        pe = pe.unsqueeze(dim=0)  # 3D tensor (1, seq_len, d_model)
        # While sending data, a buffer keeps the data temporarily
        # With this module, the buffer is saved and loaded when the model is.
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Input x + positional encoding
        # No need to back-propagate
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        # To avoid overfitting, change some of elements into O
        return self.dropout(x)

# ----------------------------------------------------------------
# 3. Layer Nomalization
# ----------------------------------------------------------------
class LayerNormalization(nn.Module):
    def __init__(self, eps: float = 10**-6) -> None:
        super.__init__()
        self.eps = eps
        # multiplied
        # [1.0]
        #  ex) torch.ones(3) --> [1.0 1.0 1.0]
        self.alpha = nn.Parameter(torch.ones(1))
        # Addded
        # [0.0]
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x_size = (seq_len, d_model)
        # the mean of values in the last dimension
        mean = x.mean(dim=-1, keepdim=True) # (seq_len, 1)
        std = x.std(dim=-1, keepdim=True)   # (seq_len, 1)
        # (x - mean) => broadcasting
        return self.alpha * (x-mean) / (std + self.eps) + self.bias


# ----------------------------------------------------------------
# 4. Feed Forward
# ----------------------------------------------------------------
class FeedForwardBlock(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))
    

# ----------------------------------------------------------------
# 5. Multi-Head Attention
# ----------------------------------------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super.__init__()
        self.d_model = d_model
        # the number of heads
        # ex) 8
        self.h = h
        # AssertionError
        # When not meeting the condition , it stops
        assert d_model % h == 0, 'd_model is not divisible by h'

        # the number of dimensions that each head handles.
        # ex) 512 / 8 = 64
        self.d_k = d_model // h

        # Linear layers for making Q, K, V
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        # A linear layer to sum up the results from the heads
        self.w_o = nn.Linear(d_model, d_model)
        # to avoid overfitting
        self.dropout = nn.Dropout(dropout)


    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]   # (batch, seq_len, d_model)
        # @: matrix multiplication
        # .transpose(-2, -1): swap the second-to-last dimension with the last dimension
        # for scaling
        # How relevant between two tokens
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k) #(batch, h, seq_len, seq_len)

        # If mask is specified
        if mask is not None:
            # ex) mask = -1e9, then fill -1e9 instead of 0
            attention_scores.masked_fill_(mask==0, -1e9)

        # attention scores -> probability distribution (0~1)
        attention_scores = attention_scores.softmax(dim=-1)  # (batch, seq_len, seq_len)
        # If dropout is provided
        if dropout is not None:
            # apply dropout to attention_scores
            attention_scores = dropout(attention_scores)
        return (attention_scores @ value), attention_scores
    
    def forward(self, q, k, v, mask):
        # 1. Q, K, V -> project into d_k, d_k, d_v dimension
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        # 2. Q, K, V separate into #heads
        # (batch, seq_len, d_model) -> (batch, seq_len, h, d_k) -> (batch, h, seq_len, d_k)
        # in order for each head to compute independently and paralle
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # 3. the actual attention scores
        x, self.attention_scores = MultiHeadAttention.attention(query, key, value, mask, self.dropout)

        # 4. sum all the heads
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)
        
        # 5. final output
        return self.w_o(x)



        
